In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
def load_and_preprocess_data(filepath):
    df = pd.read_csv('data/rental_info.csv')
    
    # Calculate rental length in days
    df["rental_length"] = pd.to_datetime(df["return_date"]) - pd.to_datetime(df["rental_date"])
    df["rental_length_days"] = df["rental_length"].dt.days

    # Create dummy variables
    df["deleted_scenes"] = np.where(df["special_features"].str.contains("Deleted Scenes"), 1, 0)
    df["behind_the_scenes"] = np.where(df["special_features"].str.contains("Behind the Scenes"), 1, 0)

    # Drop irrelevant columns
    drop_cols = ["special_features", "rental_length", "rental_length_days", "rental_date", "return_date"]
    X = df.drop(drop_cols, axis=1)
    y = df["rental_length_days"]
    return train_test_split(X, y, test_size=0.2, random_state=9)

def run_lasso_feature_selection(X_train, y_train, X_test):
    lasso = Lasso(alpha=0.3, random_state=9)
    lasso.fit(X_train, y_train)
    coef = lasso.coef_
    selected_cols = coef > 0
    return X_train.iloc[:, selected_cols], X_test.iloc[:, selected_cols]

def evaluate_linear_regression(X_train, y_train, X_test, y_test):
    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return mean_squared_error(y_test, predictions)

def tune_random_forest(X_train, y_train):
    param_dist = {
        'n_estimators': np.arange(1, 101, 1),
        'max_depth': np.arange(1, 11, 1)
    }
    rf = RandomForestRegressor(random_state=9)
    search = RandomizedSearchCV(rf, param_distributions=param_dist, cv=5, random_state=9)
    search.fit(X_train, y_train)
    return search.best_params_

def evaluate_random_forest(X_train, y_train, X_test, y_test, best_params):
    model = RandomForestRegressor(**best_params, random_state=9)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return model, mean_squared_error(y_test, predictions)

def main():
    X_train, X_test, y_train, y_test = load_and_preprocess_data("rental_info.csv")

    # Feature selection using Lasso
    X_lasso_train, X_lasso_test = run_lasso_feature_selection(X_train, y_train, X_test)
    
    # Evaluate Linear Regression
    mse_lasso = evaluate_linear_regression(X_lasso_train, y_train, X_lasso_test, y_test)
    print(f"Lasso + Linear Regression MSE: {mse_lasso:.4f}")

    # Random Forest with Hyperparameter Tuning
    best_params = tune_random_forest(X_train, y_train)
    best_model, mse_rf = evaluate_random_forest(X_train, y_train, X_test, y_test, best_params)
    print(f"Random Forest MSE: {mse_rf:.4f}")

    # Final Model
    if mse_rf < mse_lasso:
        print("Best model: Random Forest")
    else:
        print("Best model: Linear Regression with Lasso-selected features")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'rental_info.csv'